In [0]:
# =============================================================
# CELDA 1: GOLD — KPIs + Resumen de Mercado
# =============================================================
import json
from pyspark.sql.functions import col, avg, count, round, desc, when, current_timestamp

# Credenciales
try:
    with open("aws_secrets.json", "r") as f:
        config = json.load(f)
except FileNotFoundError:
    raise SystemExit("❌ aws_secrets.json no encontrado.")

AWS_KEY = config["aws_access_key"]
AWS_SECRET = config["aws_secret_key"]
BUCKET = "bronce-scrap-date"

S3_OPTS = {
    "fs.s3a.access.key": AWS_KEY,
    "fs.s3a.secret.key": AWS_SECRET,
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

# --- Leer Silver Deduped (output de 02b_Dedup_Cross_Portal) ---
# Fallback a master_inmuebles si master_deduped no existe aún
import boto3

s3_check = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1",
)
try:
    s3_check.list_objects_v2(
        Bucket=BUCKET, Prefix="silver/master_deduped/_delta_log/", MaxKeys=1
    )["Contents"]
    ruta_silver = f"s3a://{BUCKET}/silver/master_deduped/"
    print("📖 Leyendo Silver DEDUPED (cross-portal)")
except (KeyError, Exception):
    ruta_silver = f"s3a://{BUCKET}/silver/master_inmuebles/"
    print("📖 Leyendo Silver original (dedup no ejecutado aún)")

reader = spark.read.format("delta")
for k, v in S3_OPTS.items():
    reader = reader.option(k, v)
df_silver = reader.load(ruta_silver)

print(f"   {df_silver.count()} registros")
print(f"   Columnas: {df_silver.columns}")

# --- KPI Detalle: Precio por m² ---
df_gold = (
    df_silver
    .filter((col("precio_num").isNotNull()) & (col("area_m2") > 10))
    .withColumn("precio_m2", round(col("precio_num") / col("area_m2"), 0))
    .withColumn("gold_processed_at", current_timestamp())
)

print(f"\n📊 Gold detalle: {df_gold.count()} inmuebles con KPI")
display(
    df_gold.select(
        "titulo", "ubicacion_raw", "precio_num", "area_m2", "precio_m2",
        "habitaciones", "banos", "garajes", "tipo_inmueble", "estado_inmueble"
    ).limit(10)
)

# --- Resumen Agregado por Barrio ---
df_resumen = (
    df_gold.groupBy("ubicacion_raw")
    .agg(
        count("id_original").alias("total_ofertas"),
        round(avg("precio_num"), 0).alias("precio_promedio"),
        round(avg("area_m2"), 0).alias("area_promedio"),
        round(avg("precio_m2"), 0).alias("valor_m2_promedio"),
        round(avg("habitaciones"), 1).alias("habitaciones_promedio"),
        round(avg("banos"), 1).alias("banos_promedio"),
    )
    .filter(col("total_ofertas") >= 2)
    .orderBy(desc("valor_m2_promedio"))
)

print(f"\n🏆 Barrios con >= 2 ofertas: {df_resumen.count()}")
display(df_resumen)

In [0]:
# =============================================================
# CELDA 2: GUARDAR — Gold Resumen (Delta) + Gold Detalle (Parquet)
# =============================================================

# Gold Resumen: agregación por barrio (para dashboard Streamlit)
ruta_resumen = f"s3a://{BUCKET}/gold/mercado_resumen/"
writer = df_resumen.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_resumen)
print(f"💾 Gold resumen: {ruta_resumen}")

# Gold Detalle: todos los inmuebles con KPIs (para ML + App)
ruta_app = f"s3a://{BUCKET}/gold/app_inmuebles/"
writer = df_gold.coalesce(1).write.format("parquet").mode("overwrite")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_app)
print(f"🚀 Gold detalle: {ruta_app}")